In [24]:
import pandas as pd
import tensorflow as tf
import numpy as np

data = pd.read_csv('/content/sample_data/twitter_training.csv')
train_data = data.sample(10000, random_state = 42)

# train_data.shape

## Data Exploration
train_data['Positive'].value_counts()
train_data.isnull().sum() ## Check null value count

## Here also we have to check duplicate value
train_data.duplicated().sum()



## Now the Next Step is Data Cleaing
train_data = train_data.dropna()
train_data = train_data.drop_duplicates()
train_data = train_data.reset_index(drop=True)

# train_data.describe()

## Next step to remove unusual columns from the dataset bcoz we are working on RNN model so we need to keep only textual data
train_data.head()

train_data.drop(columns=['2401', 'Borderlands'], inplace=True)
train_data.head()

## Rename the column name
train_data.rename(
    columns={
        'Positive': 'Sentiment',
        'im getting on borderlands and i will murder you all ,': 'Tweets'
    },
    inplace=True
)

train_data = train_data[train_data['Sentiment'] != 'Irrelevant']
train_data['Sentiment'].value_counts()


## Feature Encoding
from sklearn.preprocessing import LabelEncoder
label = LabelEncoder()

train_data['Sentiment'] = label.fit_transform(train_data['Sentiment'])

from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer = Tokenizer(oov_token='Nothing')


tokenizer.fit_on_texts(train_data['Tweets'])
# train_data.head()
# tokenizer.word_index ## check integer encoding
## tokenizer.word_counts ## how many time word repeated , we got the idea with the help of word_counts function in tokenizer
## len(tokenizer.word_index) ## show the size of vocabulary

##


sequences = tokenizer.texts_to_sequences(train_data['Tweets'])

# len(sequences[0])  ##  Total len = 8
# len(sequences[1])   ## Total len = 29
# len(sequences[2])  ##  Total len = 1

## Problem Statement: each tweet becomes a sequence of numbers, but different tweets have different numbers of words, so the sequence lengths are different.
##Neural networks such as ANN/RNN generally need inputs in a consistent shape when you create a batch

## How to solve these type of problem , we can apply padding from the our side

from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import SimpleRNN, Dense, Input,Embedding
from tensorflow.keras.models import Sequential

X = pad_sequences(sequences, padding='post')

# X =  sequences

Y = train_data['Sentiment'].values




In [25]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size=0.2, random_state =42)

model = Sequential()
model.add(SimpleRNN(32, input_shape=(99,1), return_sequences=False))
model.add(Dense(3, activation='softmax'))
model.summary()
model.compile(
    optimizer='adam',
    metrics=['accuracy'],
    loss='sparse_categorical_crossentropy'
)
model.fit(X_train, Y_train, epochs=10, validation_data=(X_test,Y_test), batch_size=64)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_2 (SimpleRNN)        │ (None, 32)             │         1,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,187 (4.64 KB)

 Trainable params: 1,187 (4.64 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - accuracy: 0.3665 - loss: 1.0963 - val_accuracy: 0.3663 - val_loss: 1.0941
Epoch 2/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.3634 - loss: 1.0945 - val_accuracy: 0.3416 - val_loss: 1.0933
Epoch 3/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.3686 - loss: 1.0930 - val_accuracy: 0.3718 - val_loss: 1.0893
Epoch 4/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.3668 - loss: 1.0933 - val_accuracy: 0.3650 - val_loss: 1.0913
Epoch 5/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.3602 - loss: 1.0936 - val_accuracy: 0.3607 - val_loss: 1.0911
Epoch 6/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.3656 - loss: 1.0931 - val_accuracy: 0.3626 - val_loss: 1.0901
Epoch 7/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.3580 - loss: 1.0936 - val_accuracy: 0.3650 - val_loss: 1.0914
Epoch 8/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.3716 - loss: 1.0920 - val_accuracy: 0

Simple RNN Project :
Getting problem in Spam Classifier


In [26]:
## Here i am creating another model with embedding for better accuracy

vocab_size = len(tokenizer.word_index) + 1

model_2 = Sequential()
model_2.add(Input(shape = (99, )))
model_2.add(Embedding(input_dim= vocab_size, output_dim=64))
model_2.add(Dense(3, activation='softmax'))
model_2.summary()






Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 99, 64)         │       943,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 99, 3)          │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 943,491 (3.60 MB)

 Trainable params: 943,491 (3.60 MB)

 Non-trainable params: 0 (0.00 B)

Embedding Layer needs to know:
How many unique words are in the vocabulary
Tokenizer word_index should be +1
input_dim should be the total number of unique words + 1
output_dim = 128 → each word is converted into a 128-dimensional vector
These 128 values are learned by the neural network during training
0 is reserved for padding, which is why we add +1


Some important points related to embedding layer
1)  Trainable
2) Learned Automatically
3) Represents the meaning and context


What Embedding layers actually produces
If the input sequence shape is coming out  to be batch size sequence length


Flow should be:
Word-Index ----> Embedding Vector ----> RNN Memory Cell

Embedding simply tells what the word means and simple RNN tells how the meaning changes with time, Embedding are trainable

These embedding are task specific and words with similiar meaning moves closer as per the task.

In [29]:
model_final = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=128,
        input_length=99,
        mask_zero=True
    ),
    SimpleRNN(32),
    Dense(3, activation='softmax')
])


model_final.compile(
    optimizer='adam',
    metrics=['accuracy'],
    loss='sparse_categorical_crossentropy'
)
model_final.fit(X_train, Y_train, epochs=10, validation_data=(X_test, Y_test), batch_size=32 )

model_final.summary()
model_final.save('model.h5')


Epoch 1/10


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


203/203 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - accuracy: 0.5663 - loss: 0.9385 - val_accuracy: 0.6658 - val_loss: 0.8016
Epoch 2/10
203/203 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8814 - loss: 0.3725 - val_accuracy: 0.6652 - val_loss: 0.8121
Epoch 3/10
203/203 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9662 - loss: 0.1166 - val_accuracy: 0.6788 - val_loss: 0.8754
Epoch 4/10
203/203 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9790 - loss: 0.0633 - val_accuracy: 0.6992 - val_loss: 0.9536
Epoch 5/10
203/203 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.9815 - loss: 0.0474 - val_accuracy: 0.6930 - val_loss: 0.9999
Epoch 6/10
203/203 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9832 - loss: 0.0422 - val_accuracy: 0.6918 - val_loss: 1.0506
Epoch 7/10
203/203 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9833 - loss: 0.0385 - val_accuracy: 0.6899 - val_loss: 1.0976
Epoch 8/10
203/203 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9821 - loss: 0.0370 - val_accuracy: 0.6986 - val

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (None, 74, 128)        │     1,886,592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_5 (SimpleRNN)        │ (None, 32)             │         5,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,675,531 (21.65 MB)

 Trainable params: 1,891,843 (7.22 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 3,783,688 (14.43 MB)